In [1]:
import pandas as pd
import numpy as np
from datetime import timedelta

file_path = "Prenotazioni completo pulito.xlsx"

df = pd.read_excel(file_path)

print(df.info())
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2352 entries, 0 to 2351
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   nome               2352 non-null   object        
 1   nazione            2352 non-null   object        
 2   ota                2352 non-null   object        
 3   stanza             2352 non-null   object        
 4   data_prenotazione  2326 non-null   datetime64[ns]
 5   check_in           2352 non-null   datetime64[ns]
 6   check_out          2352 non-null   datetime64[ns]
 7   notti              2352 non-null   int64         
 8   lordo              2334 non-null   float64       
 9   netto              2333 non-null   float64       
 10  n_ospiti           1383 non-null   float64       
 11  telefono           1898 non-null   object        
 12  reservation_id     1876 non-null   object        
 13  data_addebito      1851 non-null   datetime64[ns]
 14  riferime

,nome,nazione,ota,stanza,data_prenotazione,check_in,check_out,notti,lordo,netto,n_ospiti,telefono,reservation_id,data_addebito,riferimento
0,Mirella Ruosi,Italia,Booking,London,2021-06-18,2022-06-22,2022-06-23,1,66.75,54.73,1.0,+39 3473100694,3727057370,2022-06-22,2022
1,Mirella Ruosi,Italia,Booking,Rome,2021-06-18,2022-06-22,2022-06-23,1,66.75,54.73,2.0,+39 3473100694,3727057370,2022-06-22,2022
2,Mirella Ruosi,Italia,Booking,Paris,2021-06-18,2022-06-22,2022-06-23,1,66.75,54.73,2.0,+39 3473100694,3727057370,2022-06-22,2022
3,Margaret Bickley,Regno Unito,Booking,London,2021-08-07,2022-08-10,2022-08-21,0,0.00,0.00,NaN,+44 7989 263574,2927704444,2022-08-03,2022
4,Raffaele Napolitano,Italia,Booking,London,2021-10-09,2021-11-10,2021-11-11,1,55.80,45.76,2.0,+39 338 444 0002,2626561264,2021-11-11,2022


In [2]:
# ======================================
# 1. LETTURA FILE
# ======================================
df = pd.read_excel("Prenotazioni completo pulito.xlsx")

# Uniformo i nomi colonne principali
df = df.rename(columns={
    'nome': 'nome',
    'nazione': 'nazionalita',
    'ota': 'ota',
    'stanza': 'stanza',
    'data_prenotazione': 'data_prenotazione',
    'check_in': 'check_in',
    'check_out': 'check_out',
    'notti': 'notti',
    'lordo': 'lordo',
    'netto': 'netto',
    'n_ospiti': 'n_ospiti',
    'telefono': 'telefono',
    'reservation_id': 'reservation_id'
})

# ======================================
# 2. PULIZIA BASE  
# ======================================

# Date → formato datetime
for col in ['data_prenotazione', 'check_in', 'check_out']:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Importi numerici: mantiene i punti decimali già presenti, converte virgole in punti, rimuove simboli valuta e spazi
for col in ['lordo', 'netto']:
    df[col] = (
        df[col].astype(str)
        .str.replace('€', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    # Conversione in float, coerente per Power BI
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Numero telefoni senza spazio
if 'telefono' in df.columns:
    df['telefono'] = df['telefono'].astype(str).str.replace(' ', '')

# Colonne numeriche (notti, ospiti)
for col in ['notti', 'n_ospiti']:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(int)

# Rimuove spazi, uniforma spazi interni, case, caratteri "strani"
df['stanza'] = (df['stanza'].astype(str)
                  .str.strip()
                  .str.replace(r'\s+', ' ', regex=True)
                  .str.normalize('NFKC')
                  .str.title())

# Mappa alias/refusi ai 3 nomi ufficiali
alias = {
    'London': 'London', 'Londra': 'London', 'London Room': 'London',
    'Paris': 'Paris', 'Parigi': 'Paris', 'Paris Room': 'Paris',
    'Rome': 'Rome', 'Roma': 'Rome', 'Rome Room': 'Rome'
}
df['stanza'] = df['stanza'].map(lambda x: alias.get(x, x))


In [3]:
# ======================================
# 3. STATUS & IMPORTI CANCELLATI
# ======================================
df['id_status'] = np.where(df['notti'] == 0, 1, 0)  # 0=Confermata, 1=Cancellata
df.loc[df['id_status'] == 1, ['lordo', 'netto']] = 0.0

# ======================================
# 4. RESERVATION_ID MANCANTI
# ======================================
# Genera ID univoco per le prenotazioni senza reservation_id
mask_missing = df['reservation_id'].isna() | (df['reservation_id'].astype(str).str.strip() == "")
if mask_missing.any():
    df.loc[mask_missing, 'reservation_id'] = (
        "AUTO_" + df.loc[mask_missing, 'nome'].astype(str) + "_" +
        df.loc[mask_missing, 'stanza'].astype(str) + "_" +
        df.loc[mask_missing, 'check_in'].dt.strftime('%Y%m%d') + "_" +
        df.loc[mask_missing, 'check_out'].dt.strftime('%Y%m%d')
    )

***
### DIMENSION TABLES:

In [4]:
# D_STATUS
d_status = pd.DataFrame({
    'id_status': [0, 1],
    'status': ['Confermata', 'Cancellata']
})

# D_OTA
d_ota = df[['ota']].dropna().drop_duplicates().reset_index(drop=True)
d_ota['id_ota'] = range(1, len(d_ota) + 1)
d_ota = d_ota[['id_ota', 'ota']]

# D_STANZA
d_stanza = df[['stanza']].dropna().drop_duplicates().reset_index(drop=True)
d_stanza['id_stanza'] = range(1, len(d_stanza)+1)
d_stanza['capienza'] = 2
d_stanza = d_stanza[['id_stanza', 'stanza', 'capienza']]

# D_CLIENTE
d_cliente = df[['nome', 'telefono', 'nazionalita']].drop_duplicates().reset_index(drop=True)
d_cliente['id_cliente'] = range(1, len(d_cliente) + 1)
d_cliente = d_cliente[['id_cliente', 'nome', 'telefono', 'nazionalita']]

***
### FACT TABLES:

In [5]:
# F_PRENOTAZIONI
f_pre = df.copy()

# Aggiungo surrogate key
f_pre.insert(0, 'id_prenotazione_riga', range(1, len(f_pre) + 1))

# Join per aggiungere gli ID FK (cliente, ota, stanza)
f_pre = f_pre.merge(d_cliente, on=['nome', 'telefono', 'nazionalita'], how='left')
f_pre = f_pre.merge(d_ota, on='ota', how='left')
f_pre = f_pre.merge(d_stanza, on='stanza', how='left')

# Seleziona e riordina colonne
f_pre = f_pre[['id_prenotazione_riga', 'reservation_id', 'id_cliente', 'id_ota', 'id_stanza',
               'id_status', 'data_prenotazione', 'check_in', 'check_out',
               'notti', 'n_ospiti', 'lordo', 'netto']]

# F_NOTTI
# Solo confermate con date valide e notti > 0
df_ok = df[(df['id_status'] == 0) &
           df['check_in'].notna() &
           df['check_out'].notna() &
           (df['notti'] > 0)].copy()

# Ripeti ogni riga tante volte quante sono le notti
f_notti = df_ok.loc[df_ok.index.repeat(df_ok['notti'])].copy()

# Offset (0..n-1) per ogni riga originale
f_notti['offset'] = f_notti.groupby(level=0).cumcount()

# Data della notte = check_in + offset giorni
f_notti['data_notte'] = f_notti['check_in'] + pd.to_timedelta(f_notti['offset'], unit='D')

# Importi per notte (divisione per notti)
f_notti['lordo_notte'] = f_notti['lordo'] / f_notti['notti']
f_notti['netto_notte'] = f_notti['netto'] / f_notti['notti']
f_notti['occupied'] = 1

# Aggancio alle dimensioni (FK)
f_notti = f_notti.merge(d_cliente[['id_cliente','nome','telefono','nazionalita']],
                        on=['nome','telefono','nazionalita'], how='left')
f_notti = f_notti.merge(d_ota[['id_ota','ota']], on='ota', how='left')
f_notti = f_notti.merge(d_stanza[['id_stanza','stanza']], on='stanza', how='left')

# 7) PK e colonne finali (ordinate)
f_notti = f_notti.reset_index(drop=True)
f_notti.insert(0, 'id_notte', range(1, len(f_notti) + 1))

f_notti = f_notti[['id_notte',
                   'reservation_id',
                   'id_cliente', 'id_ota', 'id_stanza',
                   'stanza', 'data_notte',
                   'lordo_notte', 'netto_notte',
                   'occupied', 'n_ospiti']]

In [9]:
# ======================================
# Check finale
# ======================================
print("✅ ESPORTAZIONE COMPLETATA!")
print("• D_cliente:", len(d_cliente))
print("• D_ota:", len(d_ota))
print("• D_stanza:", len(d_stanza))
print("• D_status:", len(d_status))
print("• F_prenotazioni:", len(f_pre))
print("• F_notti:", len(f_notti))
print("💰 Totale netto prenotazioni confermate:", round(df[df['id_status'] == 0]['netto'].sum(), 2))
print("💰 Totale netto notti:", round(f_notti['netto_notte'].sum(), 2))

✅ ESPORTAZIONE COMPLETATA!
• D_cliente: 2006
• D_ota: 4
• D_stanza: 3
• D_status: 2
• F_prenotazioni: 2352
• F_notti: 3755
💰 Totale netto prenotazioni confermate: 315681.21
💰 Totale netto notti: 315681.21


In [10]:
# ======================================
# CONFRONTO F_PRENOTAZIONI vs F_NOTTI
# ======================================

# Totali F_prenotazioni (solo confermate)
pre_tot_notti = df[df['id_status'] == 0]['notti'].sum()
pre_tot_lordo = df[df['id_status'] == 0]['lordo'].sum()
pre_tot_netto = df[df['id_status'] == 0]['netto'].sum()

# Totali F_notti
notti_tot_notti = f_notti.shape[0]              # ogni riga = una notte
notti_tot_lordo = f_notti['lordo_notte'].sum()
notti_tot_netto = f_notti['netto_notte'].sum()

# Stampa confronto
print("==============================================")
print("📊 CONFRONTO TOTALE – F_PRENOTAZIONI vs F_NOTTI")
print("==============================================")
print(f"🏨  F_PRENOTAZIONI")
print(f"   • Notti totali: {pre_tot_notti:,}")
print(f"   • Lordo totale: €{pre_tot_lordo:,.2f}")
print(f"   • Netto totale: €{pre_tot_netto:,.2f}")
print("----------------------------------------------")
print(f"🛏️  F_NOTTI")
print(f"   • Notti totali: {notti_tot_notti:,}")
print(f"   • Lordo totale: €{notti_tot_lordo:,.2f}")
print(f"   • Netto totale: €{notti_tot_netto:,.2f}")
print("----------------------------------------------")

# Differenze
diff_notti = notti_tot_notti - pre_tot_notti
diff_lordo = notti_tot_lordo - pre_tot_lordo
diff_netto = notti_tot_netto - pre_tot_netto

print("🔁 DIFFERENZE")
print(f"   • Notti: {diff_notti:+,}")
print(f"   • Lordo: €{diff_lordo:,.2f}")
print(f"   • Netto: €{diff_netto:,.2f}")
print("==============================================")

📊 CONFRONTO TOTALE – F_PRENOTAZIONI vs F_NOTTI
🏨  F_PRENOTAZIONI
   • Notti totali: 3,755
   • Lordo totale: €390,063.53
   • Netto totale: €315,681.21
----------------------------------------------
🛏️  F_NOTTI
   • Notti totali: 3,755
   • Lordo totale: €390,063.53
   • Netto totale: €315,681.21
----------------------------------------------
🔁 DIFFERENZE
   • Notti: +0
   • Lordo: €-0.00
   • Netto: €0.00


In [25]:
# ======================================
# ESPORTAZIONE DI TUTTE LE TABELLE
# ======================================

# Percorso di salvataggio
path = "./"

# Dimension Tables
d_cliente.to_csv(path + "D_cliente.csv", index=False, encoding="utf-8-sig")
d_ota.to_csv(path + "D_ota.csv", index=False, encoding="utf-8-sig")
d_stanza.to_csv(path + "D_stanza.csv", index=False, encoding="utf-8-sig")
d_status.to_csv(path + "D_status.csv", index=False, encoding="utf-8-sig")

# Fact Tables
f_pre.to_csv(path + "F_prenotazioni.csv", index=False, encoding="utf-8-sig")
f_notti.to_csv(path + "F_notti.csv", index=False, encoding="utf-8-sig")

print("==============================================")
print("✅ ESPORTAZIONE COMPLETATA!")
print("==============================================")
print(f"• D_cliente:       {len(d_cliente):,} righe")
print(f"• D_ota:           {len(d_ota):,} righe")
print(f"• D_stanza:        {len(d_stanza):,} righe")
print(f"• D_status:        {len(d_status):,} righe")
print(f"• F_prenotazioni:  {len(f_pre):,} righe")
print(f"• F_notti:         {len(f_notti):,} righe")

✅ ESPORTAZIONE COMPLETATA!
• D_cliente:       2,006 righe
• D_ota:           4 righe
• D_stanza:        3 righe
• D_status:        2 righe
• F_prenotazioni:  2,352 righe
• F_notti:         3,755 righe


In [24]:
df.to_csv("f_pe.csv", index=False, sep=';', decimal=',', encoding='utf-8-sig')